# Phase 7: node2vec Embedding Training

This notebook trains node2vec embeddings on the KG using biased random walks.

## Processing Flow:
1. **Initial State** (from Phase 5-6):
   - Item nodes: Initialized with saved embeddings (random fallback if missing)
   - Attribute nodes (Category, Brand, Genre, Artist): Zero embeddings
   - User nodes: Zero embeddings

2. **Biased Random Walks**:
   - Walk length: 10 hops per walk
   - Num walks: 20 walks per node
   - p, q parameters control exploration vs exploitation
   - Weight-aware: Uses temporal decay weights on user-item edges

3. **Skip-gram Optimization** (Word2Vec):
   - Window size: 5 (context window in random walks)
   - Embeddings refined to predict neighboring nodes in walks
   - Nodes appearing together in walks become similar

4. **Embedding Refinement**:
   - Initial graph structure (all nodes) + metadata relationships
   - Nodes sharing metadata → similar embeddings
   - Related users → similar embeddings
   - Learned structural & semantic similarity

**Key hyperparameters**:
- `walk_length`: Length of each random walk
- `num_walks`: Number of walks per node (higher = better quality)
- `p`, `q`: Return and in-out parameters for BFS/DFS balance
- `window`: Skip-gram context window size

**Run Time**: ~15 mins
**Input**: KG from Phase 5-6
**Output**: node2vec model + refined embeddings


In [1]:
# Cell 1: Install dependencies
!pip install -q node2vec networkx torch gensim pandas numpy scikit-learn tqdm -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy

In [2]:
pip install --upgrade --force-reinstall numpy pandas

  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached pandas-3.0.2-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: python-dat

In [3]:
# Cell 2: Imports
import os
import pickle
import pandas as pd
import numpy as np
import networkx as nx
from node2vec import Node2Vec
from gensim.models import Word2Vec
from sklearn.preprocessing import normalize
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Environment detection - local vs Kaggle
BASE_DIR = os.getcwd()
if os.path.exists('/kaggle/input/datasets/chandrimanandi/phase-5-6-results'):
    # Real Kaggle environment
    INPUT_DIR = '/kaggle/input/datasets/chandrimanandi/phase-5-6-results'
    OUTPUT_DIR = '/kaggle/working'
else:
    # Local environment
    INPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-5-6-results')
    OUTPUT_DIR = os.path.join(BASE_DIR, 'output', 'phase-7-results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

SEED = 42
np.random.seed(SEED)
print("Setup complete")

INPUT_DIR: /kaggle/input/datasets/chandrimanandi/phase-5-6-results
OUTPUT_DIR: /kaggle/working
Setup complete


In [4]:
# Cell 3: Load KG
print("Loading KG...")

with open(f"{INPUT_DIR}/knowledge_graph.pkl", 'rb') as f:
    G = pickle.load(f)

with open(f"{INPUT_DIR}/kg_embeddings.pkl", 'rb') as f:
    kg_emb_data = pickle.load(f)

embedding_dim = kg_emb_data['embedding_dim']

print(f"KG loaded: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Embedding dim: {embedding_dim}")

Loading KG...
KG loaded: 9,112 nodes, 128,656 edges
Embedding dim: 384


## node2vec Configuration

In [5]:
# Cell 4: node2vec hyperparameters
print("\n" + "="*60)
print("PHASE 7: NODE2VEC TRAINING")
print("="*60)

# Config
N2V_DIM = embedding_dim  # Match KG initialization dimension
N2V_WALK_LEN = 10         # Length of each random walk (hop distance)
N2V_NUM_WALKS = 20        # Number of walks per node (sample efficiency)
N2V_P = 1.0               # Return parameter (1.0 = uniform, no bias to return)
N2V_Q = 0.5               # In-out parameter (< 1 = prefer DFS/local structure, explore communities)
N2V_WORKERS = 4           # Parallel processing threads
N2V_WINDOW = 5            # Skip-gram window (context window in random walks)
N2V_MIN_COUNT = 1         # Minimum word count threshold

print("\n" + "="*60)
print("HYPERPARAMETER CONFIGURATION")
print("="*60)
print(f"Embedding dimension: {N2V_DIM} (matches initial semantic embeddings)")
print(f"Walk length: {N2V_WALK_LEN} (hops per walk)")
print(f"Num walks: {N2V_NUM_WALKS} (walks per node for better coverage)")
print(f"p (return): {N2V_P} (1.0 = equal probability to return)")
print(f"q (in-out): {N2V_Q} (< 1 prefers local structure: DFS behavior)")
print(f"Window size: {N2V_WINDOW} (context window for skip-gram)")
print("\nInterpretation:")
print("- q=0.5 encourages exploring connected communities (genre, brand groups)")
print("- Random walks capture: interaction patterns + metadata relationships")
print("- Skip-gram learns to embed nodes that appear together in walks")
print("- Result: Items of same genre/brand become semantically closer")



PHASE 7: NODE2VEC TRAINING

HYPERPARAMETER CONFIGURATION
Embedding dimension: 384 (matches initial semantic embeddings)
Walk length: 10 (hops per walk)
Num walks: 20 (walks per node for better coverage)
p (return): 1.0 (1.0 = equal probability to return)
q (in-out): 0.5 (< 1 prefers local structure: DFS behavior)
Window size: 5 (context window for skip-gram)

Interpretation:
- q=0.5 encourages exploring connected communities (genre, brand groups)
- Random walks capture: interaction patterns + metadata relationships
- Skip-gram learns to embed nodes that appear together in walks
- Result: Items of same genre/brand become semantically closer


In [6]:
# Cell 5: Initialize node2vec
print("\nInitializing node2vec...")

# Create undirected version for preprocessing
G_undirected = G.to_undirected()

node2vec = Node2Vec(
    G,
    dimensions=N2V_DIM,
    walk_length=N2V_WALK_LEN,
    num_walks=N2V_NUM_WALKS,
    p=N2V_P,
    q=N2V_Q,
    workers=N2V_WORKERS,
    weight_key='weight',  # Use temporal decay weights
    seed=SEED,
    quiet=False
)

print("node2vec initialized")


Initializing node2vec...


Computing transition probabilities:   0%|          | 0/9112 [00:00<?, ?it/s]

Generating walks (CPU: 3): 100%|██████████| 5/5 [01:55<00:00, 23.11s/it]


node2vec initialized


Generating walks (CPU: 4): 100%|██████████| 5/5 [01:53<00:00, 22.73s/it]


In [8]:
# Cell 6: Train embeddings
from gensim.models.callbacks import CallbackAny2Vec
import time

class EpochProgressCallback(CallbackAny2Vec):
    """Logs loss and time after each epoch."""
    def __init__(self, total_epochs):
        self.total   = total_epochs
        self.epoch   = 0
        self.t_start = None

    def on_epoch_begin(self, model):
        self.t_start = time.time()

    def on_epoch_end(self, model):
        self.epoch += 1
        elapsed = time.time() - self.t_start
        # gensim exposes running loss via get_latest_training_loss()
        try:
            loss = model.get_latest_training_loss()
        except Exception:
            loss = float('nan')
        bar_filled = int(20 * self.epoch / self.total)
        bar = '█' * bar_filled + '░' * (20 - bar_filled)
        print(f"  Epoch {self.epoch:2d}/{self.total} [{bar}] "
              f"loss={loss:.2f}  ({elapsed:.1f}s)", flush=True)

print("\nTraining node2vec (generating random walks + Skip-gram)...")
print("(This may take 10-15 minutes)\n")

callback = EpochProgressCallback(total_epochs=10)

n2v_model = node2vec.fit(
    window=N2V_WINDOW,
    min_count=N2V_MIN_COUNT,
    batch_words=4,
    workers=N2V_WORKERS,
    seed=SEED,
    epochs=10,
    compute_loss=True,    # required for loss tracking
    callbacks=[callback]
)

print("\n✓ node2vec training complete")
print(f"Vocabulary size: {len(n2v_model.wv)}")


Training node2vec (generating random walks + Skip-gram)...
(This may take 10-15 minutes)

  Epoch  1/10 [██░░░░░░░░░░░░░░░░░░] loss=3130128.25  (35.3s)
  Epoch  2/10 [████░░░░░░░░░░░░░░░░] loss=5683396.50  (32.9s)
  Epoch  3/10 [██████░░░░░░░░░░░░░░] loss=7972260.00  (33.6s)
  Epoch  4/10 [████████░░░░░░░░░░░░] loss=9919021.00  (33.4s)
  Epoch  5/10 [██████████░░░░░░░░░░] loss=11762299.00  (32.5s)
  Epoch  6/10 [████████████░░░░░░░░] loss=13578290.00  (33.3s)
  Epoch  7/10 [██████████████░░░░░░] loss=15374444.00  (32.8s)
  Epoch  8/10 [████████████████░░░░] loss=17175232.00  (32.7s)
  Epoch  9/10 [██████████████████░░] loss=19099880.00  (33.0s)
  Epoch 10/10 [████████████████████] loss=21004996.00  (34.6s)

✓ node2vec training complete
Vocabulary size: 9112


In [10]:
# ── Phase 7 sanity check ──────────────────────────────────────────────────────
print("=== NODE2VEC SANITY CHECK ===")

# Get n_items from the graph itself (already loaded)
n_items = max([int(n.split('_')[1]) for n in G.nodes() if n.startswith('item_')]) + 1
n_users = max([int(n.split('_')[1]) for n in G.nodes() if n.startswith('user_')]) + 1
print(f"Derived from graph — n_items: {n_items}, n_users: {n_users}")

# 1. Vocab covers items
item_nodes_in_vocab = sum(1 for i in range(n_items) if f'item_{i}' in n2v_model.wv)
print(f"Items in vocab: {item_nodes_in_vocab}/{n_items} ({100*item_nodes_in_vocab/n_items:.1f}%)")

# 2. Embeddings have variance (not collapsed)
sample_embs = np.array([n2v_model.wv[f'item_{i}']
                         for i in range(min(100, n_items))
                         if f'item_{i}' in n2v_model.wv])
print(f"Embedding std across items: {sample_embs.std():.6f}  (should be > 0.01)")

# 3. Cosine similarity between two items
e1 = n2v_model.wv['item_0']
e2 = n2v_model.wv['item_1']
sim_adjacent = np.dot(e1/np.linalg.norm(e1), e2/np.linalg.norm(e2))
print(f"Sample cosine similarity item_0↔item_1: {sim_adjacent:.4f}")

# 4. Most similar items to item_0 (qualitative check)
try:
    most_similar = n2v_model.wv.most_similar('item_0', topn=3)
    print(f"Most similar to item_0: {most_similar}")
except Exception as e:
    print(f"most_similar check skipped: {e}")

# Final verdict
if sample_embs.std() < 0.01:
    print("\n⚠️  WARNING: embeddings have very low variance — may not have converged")
elif item_nodes_in_vocab < n_items * 0.9:
    print(f"\n⚠️  WARNING: only {item_nodes_in_vocab}/{n_items} items in vocab — some items had no interactions")
else:
    print("\n✅ node2vec embeddings look healthy")

=== NODE2VEC SANITY CHECK ===
Derived from graph — n_items: 3568, n_users: 5541
Items in vocab: 3568/3568 (100.0%)
Embedding std across items: 0.141666  (should be > 0.01)
Sample cosine similarity item_0↔item_1: 0.2237
Most similar to item_0: [('user_541', 0.5456119179725647), ('user_5331', 0.5359488725662231), ('user_4041', 0.5200874209403992)]

✅ node2vec embeddings look healthy


In [11]:
# Cell 7: Extract embeddings
print("\nExtracting embeddings...")
print("\nEmbedding Refinement Summary:")
print("- Item nodes: Started with semantic review embeddings → refined by graph structure")
print("- Attribute nodes: Started with zero vectors → learned via random walks")
print("- User nodes: Started with zero vectors → learned via random walk patterns")
print("- Shared metadata causes embeddings to align (same genre → closer)")

# Build node embedding matrices
n_nodes = G.number_of_nodes()

# Map node names to indices
node_list = sorted(G.nodes())
node_to_idx = {node: i for i, node in enumerate(node_list)}

node_embeddings = np.zeros((n_nodes, N2V_DIM))

for node in tqdm(G.nodes(), desc="Extracting node embeddings", total=n_nodes):
    try:
        # Get embedding from node2vec model
        emb = np.array(n2v_model.wv[node])
    except KeyError:
        # Fallback for unseen nodes (shouldn't happen)
        emb = np.random.randn(N2V_DIM) * 0.01
    
    idx = node_to_idx[node]
    node_embeddings[idx] = emb

# Normalize for cosine similarity later
node_embeddings_normalized = normalize(node_embeddings, norm='l2')

print(f"\nNode embeddings shape: {node_embeddings.shape}")
print(f"Embedding range: [{node_embeddings.min():.4f}, {node_embeddings.max():.4f}]")
print(f"Normalized L2 norms: min={np.linalg.norm(node_embeddings_normalized[0]):.4f}, max={np.linalg.norm(node_embeddings_normalized[-1]):.4f}")



Extracting embeddings...

Embedding Refinement Summary:
- Item nodes: Started with semantic review embeddings → refined by graph structure
- Attribute nodes: Started with zero vectors → learned via random walks
- User nodes: Started with zero vectors → learned via random walk patterns
- Shared metadata causes embeddings to align (same genre → closer)


Extracting node embeddings: 100%|██████████| 9112/9112 [00:00<00:00, 205351.04it/s]


Node embeddings shape: (9112, 384)
Embedding range: [-1.0660, 1.0230]
Normalized L2 norms: min=1.0000, max=1.0000


In [12]:
# Cell 8: Extract item-specific embeddings
print("\nExtracting item embeddings for efficient lookup...")

n_items_train = max([int(n.split('_')[1]) for n in G.nodes() if n.startswith('item_')]) + 1

n2v_item_embeddings = np.zeros((n_items_train, N2V_DIM))

for item_id in range(n_items_train):
    node_name = f'item_{item_id}'
    try:
        emb = np.array(n2v_model.wv[node_name])
    except KeyError:
        emb = np.random.randn(N2V_DIM) * 0.01
    n2v_item_embeddings[item_id] = emb

n2v_item_embeddings_normalized = normalize(n2v_item_embeddings, norm='l2')

print(f"Item embeddings shape: {n2v_item_embeddings.shape}")


Extracting item embeddings for efficient lookup...
Item embeddings shape: (3568, 384)


In [18]:
# Cell 9: Quality checks
print("\n" + "="*60)
print("EMBEDDING QUALITY CHECKS")
print("="*60)

# Check cosine similarities between similar items
print("\nSample cosine similarities (Random pairs):")
for _ in range(3):
    i1, i2 = np.random.choice(n_items_train, 2, replace=False)
    sim = np.dot(n2v_item_embeddings_normalized[i1], n2v_item_embeddings_normalized[i2])
    print(f"  item_{i1} ↔ item_{i2}: {sim:.4f}")

# Statistics
print(f"\nEmbedding statistics:")
print(f"  Min: {n2v_item_embeddings.min():.4f}")
print(f"  Max: {n2v_item_embeddings.max():.4f}")
print(f"  Mean: {n2v_item_embeddings.mean():.4f}")
print(f"  Std: {n2v_item_embeddings.std():.4f}")

# Check for NaN
nan_count = np.isnan(n2v_item_embeddings).sum()
print(f"  NaN values: {nan_count}")


EMBEDDING QUALITY CHECKS

Sample cosine similarities (Random pairs):
  item_779 ↔ item_1277: 0.2521
  item_3447 ↔ item_420: 0.1055
  item_1011 ↔ item_1412: 0.1872

Embedding statistics:
  Min: -1.0660
  Max: 0.9234
  Mean: 0.0016
  Std: 0.1491
  NaN values: 0


In [19]:
# Cell 10: Save outputs
print("\n" + "="*60)
print("SAVING OUTPUTS")
print("="*60)

# Save model
print("Saving node2vec model...")
with open(f"{OUTPUT_DIR}/n2v_model.pkl", 'wb') as f:
    pickle.dump(n2v_model, f)
print("✓ Model saved")

# Save embeddings
print("Saving embeddings...")
with open(f"{OUTPUT_DIR}/n2v_embeddings.pkl", 'wb') as f:
    pickle.dump({
        'node_embeddings': node_embeddings,
        'node_embeddings_normalized': node_embeddings_normalized,
        'item_embeddings': n2v_item_embeddings,
        'item_embeddings_normalized': n2v_item_embeddings_normalized,
        'node_to_idx': node_to_idx,
        'node_list': node_list,
    }, f)
print("✓ Embeddings saved")

# Save hyperparameters
import json
hparams = {
    'dimension': N2V_DIM,
    'walk_length': N2V_WALK_LEN,
    'num_walks': N2V_NUM_WALKS,
    'p': N2V_P,
    'q': N2V_Q,
    'window': N2V_WINDOW,
    'workers': N2V_WORKERS,
}

with open(f"{OUTPUT_DIR}/n2v_hyperparams.json", 'w') as f:
    json.dump(hparams, f, indent=2)
print("✓ Hyperparams saved")

print(f"\n{'='*60}")
print(f"PHASE 7 COMPLETE")
print(f"{'='*60}")
print(f"Total nodes: {n_nodes}")
print(f"Total items: {n_items_train}")
print(f"Embedding dimension: {N2V_DIM}")
print(f"\nAll files saved to: {OUTPUT_DIR}")


SAVING OUTPUTS
Saving node2vec model...
✓ Model saved
Saving embeddings...
✓ Embeddings saved
✓ Hyperparams saved

PHASE 7 COMPLETE
Total nodes: 9112
Total items: 3568
Embedding dimension: 384

All files saved to: /kaggle/working
